# Export Formats

The `exports` module converts flight plans into file formats used by
flight crews, data archives, and briefing workflows. All formats are
drop-in compatible with [MovingLines](https://github.com/samuelleblanc/fp),
NASA's standard flight planning tool.

This notebook covers:

1. Building a sample flight plan
2. Coordinate formatting and magnetic declination
3. Extracting a waypoint table
4. Pilot Excel export
5. ForeFlight CSV for iPad
6. Honeywell FMS for G-III/G-IV avionics
7. ER-2 CSV
8. ICARTT for NASA data archiving
9. KML/KMZ for Google Earth
10. GPX for GPS devices
11. Plain text
12. Full working Excel (round-trip with MovingLines)

In [ ]:
import datetime
import os
import tempfile

import pandas as pd

from hyplan import (
    DynamicAviation_B200, NASA_ER2,
    Airport, initialize_data,
    compute_flight_plan,
    FlightLine,
)
from hyplan.waypoint import Waypoint
from hyplan.units import ureg
from hyplan.geometry import (
    magnetic_declination, true_to_magnetic,
    dd_to_ddm, dd_to_ddms, dd_to_nddmm, dd_to_foreflight_oneline,
)
from hyplan.exports import (
    extract_waypoints, generate_wp_names,
    to_excel, to_pilot_excel,
    to_foreflight_csv, to_honeywell_fms, to_er2_csv,
    to_icartt, to_kml, to_gpx, to_txt,
)

# Output directory for generated files
OUT_DIR = tempfile.mkdtemp(prefix="hyplan_exports_")
print(f"Export files will be written to: {OUT_DIR}")

## 1. Build a Sample Flight Plan

We create a multi-waypoint flight plan from Edwards AFB using a
King Air B200, with altitude changes between waypoints — a realistic
scenario that exercises all export features.

In [ ]:
initialize_data(countries=["US"])
b200 = DynamicAviation_B200()
kedw = Airport("KEDW")

waypoints = [
    Waypoint(34.7, -118.2, 0.0,
             altitude_msl=ureg.Quantity(10000, "feet"), name="WP1"),
    Waypoint(34.9, -118.0, 45.0,
             altitude_msl=ureg.Quantity(20000, "feet"), name="WP2"),
    Waypoint(35.1, -118.2, 180.0,
             altitude_msl=ureg.Quantity(15000, "feet"), name="WP3"),
    Waypoint(34.8, -118.4, 270.0,
             altitude_msl=ureg.Quantity(18000, "feet"), name="WP4"),
]

plan = compute_flight_plan(
    b200, waypoints,
    takeoff_airport=kedw, return_airport=kedw,
)

# Mission metadata
TAKEOFF_TIME = datetime.datetime(2025, 6, 15, 15, 0, 0)
MISSION_NAME = "HYPLAN-DEMO"

print(f"Flight plan: {len(plan)} segments")
print(f"Aircraft: {b200.aircraft_type} ({b200.tail_number})")
print(f"Airport: {kedw.icao_code} — {kedw.name}")
print()
plan[["segment_type", "segment_name", "distance", "time_to_segment"]]

## 2. Coordinate Formatting and Magnetic Declination

Pilot exports need coordinates in various formats and magnetic headings.
These utilities live in `hyplan.geometry`.

In [ ]:
# Example point: Edwards AFB
lat, lon = kedw.latitude, kedw.longitude
print(f"Decimal degrees:   {lat:.6f}, {lon:.6f}")
print(f"DD MM (pilot):     {dd_to_ddm(lat, lon)}")
print(f"DD MM SS:          {dd_to_ddms(lat, lon)}")
print(f"NDDD MM.SS (FMS):  {dd_to_nddmm(lat, lon)}")
print(f"ForeFlight inline: {dd_to_foreflight_oneline(lat, lon)}")
print()

# Magnetic declination
dec = magnetic_declination(lat, lon)
print(f"Magnetic declination at KEDW: {dec:.2f}° (positive = east)")
print(f"True heading 090° → Magnetic: {true_to_magnetic(90.0, dec):.1f}°")
print(f"True heading 360° → Magnetic: {true_to_magnetic(360.0, dec):.1f}°")

## 3. Waypoint Table Extraction

`extract_waypoints()` converts the segment-based flight plan into a
waypoint-based table — one row per waypoint with cumulative distance,
time, altitude, heading, and speed.

In [ ]:
wps = extract_waypoints(plan)
print(f"{len(wps)} waypoints extracted from {len(plan)} segments\n")

display_cols = [
    "wp", "lat", "lon", "alt_kft", "heading",
    "speed_kt", "dist_nm", "cum_dist_nm",
    "leg_time_min", "cum_time_min", "segment_type",
]
wps[display_cols].round(2)

In [ ]:
# Waypoint naming (MovingLines-compatible 5-char names)
names = generate_wp_names(len(wps), prefix="B", date=TAKEOFF_TIME.date())
print("Generated waypoint names:")
for name, (_, wp) in zip(names, wps.iterrows()):
    print(f"  {name}  ({wp['lat']:.4f}, {wp['lon']:.4f})  {wp['alt_kft']:.1f} kft")

## 4. Pilot Excel

Simplified waypoint table for the flight crew, matching the
MovingLines `_for_pilots.xlsx` format. Includes waypoint names,
formatted coordinates, altitude, UTC time, and magnetic heading.

Three coordinate formats are supported via the `coord_format` parameter:
- `'DD MM'` — degrees and decimal minutes (default)
- `'DD MM SS'` — degrees, minutes, seconds
- `'NDDD MM.SS'` — hemisphere prefix with decimal minutes

In [ ]:
pilot_path = os.path.join(OUT_DIR, "demo_for_pilots.xlsx")
to_pilot_excel(
    plan, pilot_path,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
    coord_format="DD MM",
    include_mag_heading=True,
)
print(f"Pilot Excel: {pilot_path}")
print(f"File size: {os.path.getsize(pilot_path):,} bytes")

# Read it back to verify
df = pd.read_excel(pilot_path, skiprows=1)
df

## 5. ForeFlight CSV

Simple CSV for importing waypoints into the ForeFlight iPad app.
Also generates a companion `_oneline.txt` with all waypoints in
the compact `N3424.210/W11803.450` format for quick entry.

In [ ]:
ff_path = os.path.join(OUT_DIR, "demo_FOREFLIGHT.csv")
to_foreflight_csv(plan, ff_path, takeoff_time=TAKEOFF_TIME)

print("=== ForeFlight CSV ===")
with open(ff_path) as f:
    print(f.read())

print("=== One-liner (for quick entry) ===")
with open(ff_path.replace(".csv", "_oneline.txt")) as f:
    print(f.read())

## 6. Honeywell FMS

CSV format for Honeywell avionics on NASA G-III and G-IV aircraft.
Coordinates use the `NDD MM.SS` / `WDDD MM.SS` format.

In [ ]:
hw_path = os.path.join(OUT_DIR, "demo_Honeywell.csv")
to_honeywell_fms(plan, hw_path, takeoff_time=TAKEOFF_TIME)

print("=== Honeywell FMS CSV ===")
with open(hw_path) as f:
    print(f.read())

## 7. ER-2 CSV

Waypoint file for the NASA ER-2 high-altitude aircraft. Includes
all waypoints (no duplicate skipping) with UTC times.

In [ ]:
er2_path = os.path.join(OUT_DIR, "demo_ER2.csv")
to_er2_csv(plan, er2_path, takeoff_time=TAKEOFF_TIME)

print("=== ER-2 CSV ===")
with open(er2_path) as f:
    print(f.read())

## 8. ICARTT

NASA's standard format for airborne science data exchange (version 1001).
The flight plan is linearly interpolated at 60-second intervals to produce
a time series of lat, lon, altitude, speed, bearing, and solar zenith angle.

Filename convention: `{mission}-Flt-plan_{platform}_{YYYYMMDD}_{revision}.ict`

In [ ]:
ict_path = os.path.join(OUT_DIR, f"{MISSION_NAME}-Flt-plan_B200_{TAKEOFF_TIME:%Y%m%d}_RA.ict")
to_icartt(
    plan, ict_path,
    pi_name="R. Pavlick",
    institution="NASA JPL",
    mission_name=MISSION_NAME,
    flight_date=TAKEOFF_TIME.date(),
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    interval_seconds=60.0,
    revision="RA",
    revision_comments="RA: Planned flight track - pre-flight",
)

print(f"ICARTT file: {os.path.basename(ict_path)}")
print(f"File size: {os.path.getsize(ict_path):,} bytes\n")

# Show header and first few data lines
with open(ict_path) as f:
    lines = f.readlines()

n_header = int(lines[0].split(",")[0])
print(f"Header: {n_header} lines")
print(f"Data: {len(lines) - n_header} points at 60-second intervals\n")

print("--- Header ---")
for line in lines[:n_header]:
    print(line, end="")

print("\n--- First 5 data lines ---")
for line in lines[n_header:n_header + 5]:
    print(line, end="")

## 9. KML/KMZ for Google Earth

KML with waypoint markers and a flight path line. Both `.kml` and
`.kmz` files are generated. The `altitude_exaggeration` parameter
scales altitude in the 3D view (default 1.0 = true scale;
MovingLines uses 10).

In [ ]:
kml_path = os.path.join(OUT_DIR, "demo_flight_plan.kml")
to_kml(
    plan, kml_path,
    takeoff_time=TAKEOFF_TIME,
    altitude_exaggeration=1.0,
)

print(f"KML: {kml_path}")
print(f"KMZ: {kml_path.replace('.kml', '.kmz')}")
print(f"KML size: {os.path.getsize(kml_path):,} bytes")
print(f"KMZ size: {os.path.getsize(kml_path.replace('.kml', '.kmz')):,} bytes")

# Show a snippet of the KML
with open(kml_path) as f:
    content = f.read()
# Show first Placemark
start = content.find("<Placemark>")
end = content.find("</Placemark>") + len("</Placemark>")
print(f"\n--- First waypoint Placemark ---")
print(content[start:end])

## 10. GPX

Standard GPS exchange format. Creates a route with waypoints that
can be loaded into GPS devices and mapping applications.

In [ ]:
gpx_path = os.path.join(OUT_DIR, "demo_flight_plan.gpx")
to_gpx(
    plan, gpx_path,
    mission_name=MISSION_NAME,
    takeoff_time=TAKEOFF_TIME,
)

print(f"GPX: {gpx_path}")
print(f"File size: {os.path.getsize(gpx_path):,} bytes\n")

with open(gpx_path) as f:
    print(f.read())

## 11. Plain Text

Tab-separated text file matching the MovingLines `save2txt` format.
Note: longitude comes before latitude (opposite of the Excel format)
for MovingLines compatibility.

In [ ]:
txt_path = os.path.join(OUT_DIR, "demo_flight_plan.txt")
to_txt(plan, txt_path, takeoff_time=TAKEOFF_TIME)

print(f"TXT: {txt_path}\n")
with open(txt_path) as f:
    print(f.read())

## 12. Full Working Excel

The comprehensive 24-column format (A–X) that MovingLines uses as its
internal working format. This enables round-tripping: a MovingLines
user can open a hyplan-generated file and continue editing, or vice
versa.

Columns include both metric and imperial units, solar geometry (SZA, AZI),
cumulative distances, and timing information.

In [ ]:
excel_path = os.path.join(OUT_DIR, "demo_full.xlsx")
to_excel(
    plan, excel_path,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
)

print(f"Full Excel: {excel_path}")
print(f"File size: {os.path.getsize(excel_path):,} bytes\n")

# Read it back and show all 24 columns
df_full = pd.read_excel(excel_path)
print(f"Columns ({len(df_full.columns)}): {list(df_full.columns)}\n")
df_full

## 13. Export All at Once

In practice, you'll often want to generate all formats for a single
flight plan. Here's a helper pattern that writes everything to a
named directory.

In [ ]:
def export_all(plan, output_dir, aircraft=None, takeoff_time=None,
               mission_name="", pi_name="", institution=""):
    """Generate all export formats for a flight plan."""
    os.makedirs(output_dir, exist_ok=True)
    base = os.path.join(output_dir, mission_name or "flight_plan")
    date_str = takeoff_time.strftime("%Y%m%d") if takeoff_time else "undated"
    platform = getattr(aircraft, "aircraft_type", "unknown").replace(" ", "_")

    files = {}
    files["excel"] = f"{base}.xlsx"
    files["pilot_excel"] = f"{base}_for_pilots.xlsx"
    files["foreflight"] = f"{base}_FOREFLIGHT.csv"
    files["honeywell"] = f"{base}_Honeywell.csv"
    files["er2"] = f"{base}_ER2.csv"
    files["icartt"] = f"{base}-Flt-plan_{platform}_{date_str}_RA.ict"
    files["kml"] = f"{base}.kml"
    files["gpx"] = f"{base}.gpx"
    files["txt"] = f"{base}.txt"

    to_excel(plan, files["excel"], aircraft=aircraft,
             takeoff_time=takeoff_time, mission_name=mission_name)
    to_pilot_excel(plan, files["pilot_excel"], aircraft=aircraft,
                   takeoff_time=takeoff_time, mission_name=mission_name)
    to_foreflight_csv(plan, files["foreflight"], takeoff_time=takeoff_time)
    to_honeywell_fms(plan, files["honeywell"], takeoff_time=takeoff_time)
    to_er2_csv(plan, files["er2"], takeoff_time=takeoff_time)
    to_icartt(plan, files["icartt"], pi_name=pi_name,
              institution=institution, mission_name=mission_name,
              flight_date=takeoff_time.date() if takeoff_time else None,
              aircraft=aircraft, takeoff_time=takeoff_time)
    to_kml(plan, files["kml"], takeoff_time=takeoff_time)
    to_gpx(plan, files["gpx"], mission_name=mission_name,
           takeoff_time=takeoff_time)
    to_txt(plan, files["txt"], takeoff_time=takeoff_time)

    return files


# Run it
all_dir = os.path.join(OUT_DIR, "all_exports")
files = export_all(
    plan, all_dir,
    aircraft=b200,
    takeoff_time=TAKEOFF_TIME,
    mission_name=MISSION_NAME,
    pi_name="R. Pavlick",
    institution="NASA HQ",
)

print(f"All files written to: {all_dir}\n")
for fmt, path in files.items():
    size = os.path.getsize(path)
    print(f"  {fmt:<15s} {os.path.basename(path):<50s} {size:>8,} bytes")

## Summary

| Function | Output | Compatible with |
|----------|--------|-----------------|
| `to_excel()` | Full 24-column `.xlsx` | MovingLines `write_to_excel()` |
| `to_pilot_excel()` | Simplified pilot `.xlsx` | MovingLines `save2xl_for_pilots_xlswriter()` |
| `to_foreflight_csv()` | ForeFlight CSV + oneline `.txt` | MovingLines ForeFlight export |
| `to_honeywell_fms()` | Honeywell FMS `.csv` | MovingLines Honeywell export |
| `to_er2_csv()` | ER-2 `.csv` | MovingLines ER-2 export |
| `to_icartt()` | ICARTT v1001 `.ict` | MovingLines `write_ict()` |
| `to_kml()` | `.kml` + `.kmz` | MovingLines `save2kml()` |
| `to_gpx()` | GPX route `.gpx` | MovingLines `save2gpx()` |
| `to_txt()` | Plain text `.txt` | MovingLines `save2txt()` |

| Helper | Purpose |
|--------|---------|
| `extract_waypoints(plan)` | Convert segment GeoDataFrame to waypoint table |
| `generate_wp_names(n, prefix, date)` | 5-char MovingLines-compatible waypoint names |
| `magnetic_declination(lat, lon)` | WMM magnetic declination in degrees |
| `true_to_magnetic(heading, dec)` | True heading → magnetic heading |
| `dd_to_ddm(lat, lon)` | Decimal degrees → `DD MM.MM` |
| `dd_to_ddms(lat, lon)` | Decimal degrees → `DD MM SS.S` |
| `dd_to_nddmm(lat, lon)` | Decimal degrees → `N37 24.21` / `W122 03.45` |
| `dd_to_foreflight_oneline(lat, lon)` | Decimal degrees → `N3724.210/W12203.450` |